In [1]:
import ollama

In [2]:
[print(i) for i in ollama.list().models]

model='qwen3.5:4b' modified_at=datetime.datetime(2026, 11, 22, 14, 33, 41, 356506, tzinfo=TzInfo(10800)) digest='d8b0f5e9760cd1682034f292d7ef72ec46f432149be0df7574bf2d6e92e38c04' size=3324173934 details=ModelDetails(parent_model='model.gguf', format='gguf', family='qwen35', families=['qwen35'], parameter_size='4.2B', quantization_level='Q4_K_M')
model='llama3.1:8b' modified_at=datetime.datetime(2025, 7, 6, 13, 36, 38, 320743, tzinfo=TzInfo(10800)) digest='46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e' size=4920753328 details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='8.0B', quantization_level='Q4_K_M')


[None, None]

In [54]:
r_qwen = ollama.chat(model="qwen3:8b", messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [48]:
r = ollama.chat(model="llama3.2:3b", messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [108]:
r_qwen_mlx = ollama.chat(model="qwen3.5:9b-mlx", think=False, messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [3]:
r_qwen_mlx_4b = ollama.chat(model="qwen3.5:4b", think=False, messages=[{"role":"user",
                                        "content":"Истинно ли утверждение: 'Платон был учеником Аристотеля'?"}])

In [4]:
print(
      '\n \n QWEN 3.5 4B MLX: \n', r_qwen_mlx_4b.message.content
     )


 
 QWEN 3.5 4B MLX: 
 **Нет, это утверждение ложно.** Историческая хронология показывает, что эти两位 мыслителей не могли быть учителем и учеником друг друга.

Обратимся к датам их жизни:
*   **Платон** жил примерно с 428/427 до 348/347 гг. до н. э.
*   **Аристотель** родился около 384 г. до н. э. и умер в 322 г. до н. э.

Отсюда следует следующая последовательность событий:
1.  Платон учился под руководством Сократа (известного философа).
2.  Когда Аристотелю было около 17 лет (около 367 года до н.э.), он стал учеником Платона в Академии в Афинах на несколько лет. То есть сам **Аристотель был учеником Платона**.
3.  После смерти Платона Аристотель ушел из его круга и стал одним из крупнейших независимых философов той эпохи, а позже возглавил школу Ликеев в Стагиере, где воспитывал своих собственных учеников (например, Клеанта).

Таким образом, их отношение к вопросу было наоборот: **Аристотель был учеником Платона**, а не наоборот.


In [5]:
for response in [r_qwen_mlx_4b]:
    print("\n", response.model)
    print("Причина остановки:", response.done_reason)
    print("Входные токены:", response.prompt_eval_count)
    print("Выходные токены:", response.eval_count)
    print("Время генерации:", response.eval_duration / 1e9)
    print("Токенов/с:", response.eval_count * 1e9 / response.eval_duration)
    # print("Thinking:", len(response.message.thinking))
    # print("Content:", len(response.message.content))


 qwen3.5:4b
Причина остановки: stop
Входные токены: 32
Выходные токены: 306
Время генерации: 3.002659
Токенов/с: 101.90967405889246


In [29]:
from ollama import chat
from pathlib import Path
from pydantic import BaseModel, Field

use_model = "qwen3.5:4b"

text = Path("text.txt").read_text(encoding="utf-8")

class Triple(BaseModel):
    subject: str = Field(
    )
    predicate: str = Field(
    )
    object: str = Field(
    )


class Result(BaseModel):
    triples: list[Triple]


response = chat(
    model = use_model,
    think = False,
    messages = [
        {
            "role":"system",
            "content": (
                "Ты извлекаешь отношения из текста в виде триплетов (субъект, предикат, объект). "
                "Извлекай только явно выраженные отношения. "
                "Не добавляй информацию от себя."
            )
        },
        {
            "role": "user",
            "content": text
        }
    ],
    format=Result.model_json_schema(),
    options={"temperature": 0},
)

In [30]:
result = Result.model_validate_json(response.message.content)

print("Токенов/с:", response.eval_count * 1e9 / response.eval_duration)
print("с:", response.eval_duration / 1e9)
print("\nТекст:\n")
print(text)
print("\nОтвет:\n")

for triple in result.triples:
    print(triple)

Токенов/с: 100.90637355687262
с: 9.474129

Текст:

                Плато́н (др.-греч. Πλάτων; 428/427 или 424/423 — 348/347 до н. э.), настоящее имя Аристо́кл — афинский философ классического периода Древней Греции, основатель платонической школы мысли и Академии, первого высшего учебного заведения в западном мире.
                Он считается одним из самых важных и влиятельных людей в истории человечества[1], и ключевой фигурой в истории древнегреческой и западной философии — вместе со своим учителем Сократом и самым известным из своих учеников Аристотелем[~ 1]. Часто Платона также называют одним из основателей западной религии и духовности[2]. Так называемый неоплатонизм философов, таких как Плотин и Порфирий, сильно повлиял на христианство через таких отцов церкви как Августин. Альфред Норт Уайтхед однажды заметил: «Наиболее правдоподобная общая характеристика европейской философской традиции состоит в том, что она представляет собой серию примечаний к Платону»[3].
                